<a href="https://colab.research.google.com/github/MengOonLee/LLM/blob/main/References/LangChain/LangChainAcademy/LangChain/Foundation/CreateAgent/ipynb/1.1_prompting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Basic prompting

In [ ]:
%%bash
pip install --no-cache-dir -qU \
    langchain langgraph langchain-core langchain-community \
    langchain-openai

In [1]:
import warnings
warnings.filterwarnings("ignore")
import dotenv

_ = dotenv.load_dotenv(dotenv_path=".env", override=True)

In [6]:
import os
from langchain import chat_models

llm = chat_models.init_chat_model(
    model="qwen3.5:cloud",
    model_provider="openai",
    base_url="https://ollama.com/v1",
    api_key=os.environ["OLLAMA_API_KEY"],
    temperature=0
)

In [7]:
import time
from langchain import agents, messages

agent = agents.create_agent(model=llm)

start_time = time.time()

messages = [messages.HumanMessage(content="""
What's the capital of the moon?
""")]
response = agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

The Moon does not have a capital. It is not a country or a sovereign nation, but rather a natural satellite of Earth. It has no government, no permanent population, and no cities.
Time taken: 4.10s


In [8]:
import time
from langchain import agents, messages

system_prompt = """
You are a science fiction writer, create a capital city at the users request.
"""
scifi_agent = agents.create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [messages.HumanMessage(content="""
What's the capital of the moon?
""")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

In the timeline I'm weaving for you, the Earth's silent companion finally found its voice in the year 2147, following the signing of the Regolith Accords.

The capital of the Moon is **Seleneus**.

**Location:**
It is nestled deep within the floor of **Clavius Crater**, one of the largest impact basins on the near side. The crater walls provide natural shielding from cosmic radiation and micrometeorites, while the flat floor offered the perfect canvas for expansion.

**Architecture:**
Seleneus is not a city of steel and concrete, but of **printed regolith and translucent aluminum**. From orbit, it looks like a giant, multifaceted gemstone embedded in the gray dust. The central government buildings are housed in the **Spire of Silence**, a needle-like tower that pierces the main atmospheric d

## Few-shot examples

In [9]:
import time
from langchain import agents, messages

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia
"""
scifi_agent = agents.create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [messages.HumanMessage(content="""
What's the capital of the moon?
""")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

Lunaris
Time taken: 39.19s


## Structured prompts

In [10]:
import time
from langchain import agents, messages

system_prompt = """
You are a science fiction writer, create a space capital city
at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries
"""
scifi_agent = agents.create_agent(
    model=llm,
    system_prompt=system_prompt
)

start_time = time.time()

messages = [messages.HumanMessage(content="""
What's the capital of the moon?
""")]
response = scifi_agent.invoke(input={"messages": messages})
for m in response["messages"]:
    m.pretty_print()

end_time = time.time() - start_time
print(f"Time taken: %.2fs"%(end_time))

================================ Human Message =================================


What's the capital of the moon?

================================== Ai Message ==================================

Name: Selene Prime

Location: Shackleton Crater, Lunar South Pole

Vibe: Luminous, Isolated, Regal

Economy: Helium-3 Export, Lunar Governance, High-Orbit Tourism
Time taken: 7.90s


## Structured output

In [11]:
import time
from langchain import agents, messages
import pydantic

class CapitalInfo(pydantic.BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

system_prompt = """
You are a science fiction writer.
Create a capital city at the users request.
"""
agent = agents.create_agent(
    model=llm,
    system_prompt=system_prompt,
    response_format=CapitalInfo
)

start_time = time.time()

messages = [messages.HumanMessage(content="""
What is the capital of The Moon?
""")]
response = agent.invoke(input={"messages": messages})

for m in response["messages"]:
    m.pretty_print()

capital_info = response["structured_response"]
capital_name = capital_info.name
capital_location = capital_info.location
print(f"{capital_name} is a city located at {capital_location}")

end_time = time.time() - start_time
print(f"Time taken: %.2fs seconds"%(end_time))

================================ Human Message =================================


What is the capital of The Moon?

================================== Ai Message ==================================
Tool Calls:
  CapitalInfo (call_p35qem2j)
 Call ID: call_p35qem2j
  Args:
    name: Selene Prime
    location: Mare Tranquillitatis (Sea of Tranquility)
    vibe: Futuristic, serene, and technologically advanced with crystalline domes and low-gravity architecture
    economy: Helium-3 mining, space tourism, lunar manufacturing, and interplanetary trade hub
================================= Tool Message =================================
Name: CapitalInfo

Returning structured response: name='Selene Prime' location='Mare Tranquillitatis (Sea of Tranquility)' vibe='Futuristic, serene, and technologically advanced with crystalline domes and low-gravity architecture' economy='Helium-3 mining, space tourism, lunar manufacturing, and interplanetary trade hub'
Selene Prime is a city located at Mare 